# Phase 1 — Data Cleaning & Feature Engineering

**Source:** Strong app CSV export  
**Goal:** Transform raw, set-level workout logs into two clean, analysis-ready datasets.

**Output:**
- `data/clean/clean_sets.csv` — one row per working set, with engineered features
- `data/clean/workout_summary.csv` — one row per workout, aggregated metrics

---
### Notebook structure
1. Setup & Load
2. Exploratory Inspection
3. Filter & Clean
4. Feature Engineering
5. Workout-Level Aggregation
6. Export

## 1. Setup & Load

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
# Resolve raw file — look in data/raw first, fall back to project root
_candidates = [
    Path("../data/raw/strong_userdata.csv"),
    Path("../strong_userdata.csv"),
]
RAW_FILE = next((p for p in _candidates if p.exists()), None)
assert RAW_FILE is not None, "strong_userdata.csv not found. Place it in data/raw/ or the project root."

CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw file : {RAW_FILE.resolve()}")
print(f"Clean dir: {CLEAN_DIR.resolve()}")

In [ ]:
# Strong exports use semicolons as delimiters and wraps all values in double quotes
raw = pd.read_csv(
    RAW_FILE,
    sep=";",
    quotechar='"',
    encoding="utf-8",
    dtype=str,          # load everything as string first — we cast deliberately below
    na_values=[""],
)

print(f"Rows: {len(raw):,}  |  Columns: {raw.shape[1]}")
raw.head(3)

## 2. Exploratory Inspection

Before any transformation, understand the shape and quality of the data.

In [ ]:
raw.info()

In [ ]:
# Null count per column — helps decide which columns are usable
null_pct = (raw.isnull().sum() / len(raw) * 100).round(1)
null_pct.rename("null_%").to_frame()

In [ ]:
# 'Set Order' drives the row type — inspect every distinct value
# Numeric values = working sets; 'W' = warmup; others = metadata rows
print(raw["Set Order"].value_counts(dropna=False).to_string())

In [ ]:
# Date range of the dataset
dates = pd.to_datetime(raw["Date"], errors="coerce")
print(f"Earliest workout : {dates.min().date()}")
print(f"Latest workout   : {dates.max().date()}")
print(f"Span             : {(dates.max() - dates.min()).days} days")

In [ ]:
# Top 20 exercises by raw row count (includes warmups/notes, for orientation)
raw["Exercise Name"].value_counts().head(20)

## 3. Filter & Clean

The raw file contains three types of non-set rows that must be removed before analysis:

| `Set Order` value | Meaning |
|---|---|
| `W` | Warmup set (tracked separately by Strong) |
| `Rest Timer` | Auto-logged rest period — no weight/reps data |
| `Note` | Free-text note attached to an exercise |

We keep only rows where `Set Order` is a positive integer.

In [ ]:
# Standardise column names to snake_case before any further work
COLUMN_MAP = {
    "Workout #":        "workout_id",
    "Date":             "date",
    "Workout Name":     "workout_name",
    "Duration (sec)":   "duration_sec",
    "Exercise Name":    "exercise_name",
    "Set Order":        "set_order",
    "Weight (kg)":      "weight_kg",
    "Reps":             "reps",
    "RPE":              "rpe",
    "Distance (meters)": "distance_m",
    "Seconds":          "seconds",
    "Notes":            "notes",
    "Workout Notes":    "workout_notes",
}

df = raw.rename(columns=COLUMN_MAP)

In [ ]:
# Keep only rows where set_order is a pure integer string (e.g. '1', '2', '3')
# This is more robust than an exclusion list — it survives any new metadata row types Strong might add
is_working_set = df["set_order"].str.match(r"^\d+$", na=False)

df_sets = df.loc[is_working_set].copy()

dropped = len(df) - len(df_sets)
print(f"Rows removed (warmups, rest timers, notes): {dropped:,}")
print(f"Working sets remaining: {len(df_sets):,}")

In [ ]:
# Cast each column to its correct type
# errors='coerce' turns unparseable values into NaN rather than raising — we inspect those after
df_sets = df_sets.assign(
    date         = pd.to_datetime(df_sets["date"], errors="coerce"),
    workout_id   = pd.to_numeric(df_sets["workout_id"], errors="coerce").astype("Int64"),
    duration_sec = pd.to_numeric(df_sets["duration_sec"], errors="coerce").astype("Int64"),
    set_order    = pd.to_numeric(df_sets["set_order"], errors="coerce").astype("Int64"),
    weight_kg    = pd.to_numeric(df_sets["weight_kg"], errors="coerce"),
    reps         = pd.to_numeric(df_sets["reps"], errors="coerce").astype("Int64"),
    rpe          = pd.to_numeric(df_sets["rpe"], errors="coerce"),
)

In [ ]:
# Inspect any rows where weight or reps could not be parsed — these would skew calculations
missing_core = df_sets[df_sets["weight_kg"].isna() | df_sets["reps"].isna()]
print(f"Rows with missing weight or reps: {len(missing_core):,}")
missing_core.head(10)

In [ ]:
# Drop rows without usable weight or reps — they cannot contribute to any metric
df_sets = df_sets.dropna(subset=["weight_kg", "reps"]).copy()

# Sanity check: no negative weights or reps
assert (df_sets["weight_kg"] >= 0).all(), "Negative weight values found"
assert (df_sets["reps"] > 0).all(), "Zero or negative rep values found"

print(f"Clean working sets: {len(df_sets):,}")

### Outlier Removal — Reps

Data entry errors in Strong produce physiologically impossible rep counts (e.g. `16814`).
In weighted strength training, sets above **100 reps** are effectively impossible and indicate a typo.
We log every affected row before dropping it so the removal is transparent and reproducible.

In [ ]:
# Inspect the upper tail of the reps distribution
print(df_sets["reps"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).to_string())
print()
print("Top 10 highest-rep sets:")
print(df_sets.nlargest(10, "reps")[["date", "exercise_name", "weight_kg", "reps"]].to_string(index=False))

In [ ]:
MAX_REPS = 100  # physiological ceiling for weighted strength training sets

outliers = df_sets[df_sets["reps"] > MAX_REPS]
if len(outliers) > 0:
    print(f"Removing {len(outliers)} row(s) with reps > {MAX_REPS}:")
    print(outliers[["date", "exercise_name", "weight_kg", "reps"]].to_string(index=False))
else:
    print("No rep outliers found.")

df_sets = df_sets[df_sets["reps"] <= MAX_REPS].copy()
print(f"\nWorking sets after outlier removal: {len(df_sets):,}")

## 4. Feature Engineering

### Volume
**Volume** (`weight × reps`) is the standard measure of total mechanical work per set.
Summed across a workout it gives *total volume load* — a key proxy for training stimulus.

### Estimated 1RM (Epley Formula)
The **one-rep maximum** is the gold standard for comparing strength over time.
Because you rarely test a true 1RM, we estimate it from sub-maximal sets:

$$\hat{1RM} = \text{weight} \times \left(1 + \frac{\text{reps}}{30}\right)$$

For single-rep sets the formula slightly overestimates, so we return the raw weight instead.

In [ ]:
df_sets = df_sets.assign(
    # Total mechanical work for this set
    volume_kg = df_sets["weight_kg"] * df_sets["reps"],

    # Epley 1RM estimate — clamp to actual weight for single-rep sets
    estimated_1rm = np.where(
        df_sets["reps"] == 1,
        df_sets["weight_kg"],
        df_sets["weight_kg"] * (1 + df_sets["reps"] / 30),
    ),

    # Calendar columns — useful for grouping and trend analysis later
    year  = df_sets["date"].dt.year,
    month = df_sets["date"].dt.to_period("M").astype(str),
    week  = df_sets["date"].dt.to_period("W").astype(str),
    day_of_week = df_sets["date"].dt.day_name(),
)

In [ ]:
# Spot-check: top estimated 1RM, one entry per exercise
# Verifies the Epley formula produces plausible results across different movements
sample = (
    df_sets
    .sort_values("estimated_1rm", ascending=False)
    .drop_duplicates(subset="exercise_name")
    .head(5)
    [["date", "exercise_name", "weight_kg", "reps", "estimated_1rm"]]
)
sample

## 5. Workout-Level Aggregation

Aggregate the set-level data into one row per workout.
This `workout_summary` table is the primary input for time-series and trend analysis.

In [ ]:
workout_summary = (
    df_sets
    .groupby(["workout_id", "date", "workout_name", "duration_sec"], dropna=False)
    .agg(
        total_sets        = ("set_order",     "count"),
        total_volume_kg   = ("volume_kg",     "sum"),
        peak_estimated_1rm= ("estimated_1rm", "max"),
        unique_exercises  = ("exercise_name", "nunique"),
    )
    .reset_index()
    .assign(
        duration_min = lambda x: (x["duration_sec"] / 60).round(1),
    )
    .drop(columns="duration_sec")
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Workouts: {len(workout_summary):,}")
workout_summary.head(5)

In [ ]:
# Quick descriptive summary — validates that aggregated numbers are in a plausible range
workout_summary[["duration_min", "total_sets", "total_volume_kg", "unique_exercises"]].describe().round(1)

## 6. Export

Write both datasets to `data/clean/`. These files are the single source of truth for all downstream analysis (SQL, Power BI, statistics notebook).

In [ ]:
# Select and order columns for the sets export
SETS_COLUMNS = [
    "workout_id", "date", "workout_name", "year", "month", "week", "day_of_week",
    "exercise_name", "set_order", "weight_kg", "reps", "rpe",
    "volume_kg", "estimated_1rm",
    "notes",
]

clean_sets = df_sets[SETS_COLUMNS].sort_values(["date", "workout_id", "exercise_name", "set_order"])

clean_sets_path = CLEAN_DIR / "clean_sets.csv"
clean_sets.to_csv(clean_sets_path, index=False)
print(f"Exported {len(clean_sets):,} rows → {clean_sets_path}")

In [ ]:
workout_summary_path = CLEAN_DIR / "workout_summary.csv"
workout_summary.to_csv(workout_summary_path, index=False)
print(f"Exported {len(workout_summary):,} rows → {workout_summary_path}")

In [ ]:
# Final summary — confirm output looks correct before handing off to Phase 2
print("=== clean_sets.csv ===")
print(f"  Rows         : {len(clean_sets):,}")
print(f"  Date range   : {clean_sets['date'].min().date()} → {clean_sets['date'].max().date()}")
print(f"  Exercises    : {clean_sets['exercise_name'].nunique()}")
print()
print("=== workout_summary.csv ===")
print(f"  Rows         : {len(workout_summary):,}")
print(f"  Avg duration : {workout_summary['duration_min'].mean():.1f} min")
print(f"  Avg volume   : {workout_summary['total_volume_kg'].mean():,.0f} kg")
print(f"  Avg sets     : {workout_summary['total_sets'].mean():.1f}")